# Shared Helpers for Nominatim Job Notebooks

Common utilities for authenticating to Lakebase Autoscaling Postgres via
`WorkspaceClient` (runs in the same workspace as the Lakebase instance).

Uses the `w.postgres` API (projects / branches) for the autoscaling
Lakebase backend.

Other notebooks import these functions via `%run ./_helpers`.

In [ ]:
import os
import subprocess
import tempfile
from pathlib import Path
from databricks.sdk import WorkspaceClient


# ---------------------------------------------------------------------------
# Default resource path components – override via widget / env var as needed
# ---------------------------------------------------------------------------
DEFAULT_PROJECT_ID = "nominatim-server"
DEFAULT_BRANCH_ID = "production"


def _branch_resource_path(
    project_id: str = DEFAULT_PROJECT_ID,
    branch_id: str = DEFAULT_BRANCH_ID,
) -> str:
    """Build the full branch resource path used by the Postgres API."""
    return f"projects/{project_id}/branches/{branch_id}"


def _get_primary_endpoint(
    project_id: str = DEFAULT_PROJECT_ID,
    branch_id: str = DEFAULT_BRANCH_ID,
    w: WorkspaceClient = None,
):
    """
    Return the first compute endpoint attached to the branch.

    This is typically the primary read-write compute.  Both host lookup
    and credential generation require the full endpoint resource name
    (projects/{project_id}/branches/{branch_id}/endpoints/{endpoint_id}).
    """
    if w is None:
        w = get_workspace_client()
    branch_path = _branch_resource_path(project_id, branch_id)
    endpoints = list(w.postgres.list_endpoints(parent=branch_path))
    if not endpoints:
        raise ValueError(f"No endpoints found for branch: {branch_path}")
    return endpoints[0]


def get_workspace_client() -> WorkspaceClient:
    """
    Return a WorkspaceClient that auto-authenticates from the notebook context.

    When running inside a Databricks notebook in the same workspace as the
    Lakebase instance, WorkspaceClient() picks up credentials automatically
    (no DATABRICKS_HOST / DATABRICKS_TOKEN needed).
    """
    return WorkspaceClient()


def get_lakebase_token(
    project_id: str = DEFAULT_PROJECT_ID,
    branch_id: str = DEFAULT_BRANCH_ID,
    w: WorkspaceClient = None,
    _endpoint_name: str = None,
) -> str:
    """
    Generate a fresh Lakebase OAuth token (valid ~1 hour).

    Uses w.postgres.generate_database_credential with the full endpoint
    resource path for the autoscaling Lakebase backend.

    Args:
        project_id: The Lakebase project ID (default "nominatim-server").
        branch_id:  The branch ID (default "production").
        w: Optional pre-existing WorkspaceClient.
        _endpoint_name: Optional pre-resolved endpoint resource name to
            avoid an extra list_endpoints call.

    Returns:
        OAuth token string usable as PGPASSWORD.
    """
    if w is None:
        w = get_workspace_client()
    if _endpoint_name is None:
        _endpoint_name = _get_primary_endpoint(project_id, branch_id, w=w).name
    cred = w.postgres.generate_database_credential(endpoint=_endpoint_name)
    return cred.token


def get_branch_host(
    project_id: str = DEFAULT_PROJECT_ID,
    branch_id: str = DEFAULT_BRANCH_ID,
    w: WorkspaceClient = None,
) -> str:
    """
    Look up the connection hostname for a branch via the Postgres API.

    Lists the compute endpoints attached to the branch and returns the
    host from the first available endpoint (typically the primary read-write
    compute).

    Args:
        project_id: The Lakebase project ID (default "nominatim-server").
        branch_id:  The branch ID (default "production").
        w: Optional pre-existing WorkspaceClient.

    Returns:
        The hostname string for the branch.
    """
    if w is None:
        w = get_workspace_client()
    ep = _get_primary_endpoint(project_id, branch_id, w=w)
    return ep.status.hosts.host


def get_pg_env(
    project_id: str = DEFAULT_PROJECT_ID,
    branch_id: str = DEFAULT_BRANCH_ID,
    host: str = None,
    user: str = None,
    database: str = "nominatim",
    port: str = "5432",
    sslmode: str = "require",
    w: WorkspaceClient = None,
) -> dict:
    """
    Build a full set of PG* environment variables with a fresh token.

    If *host* is not provided it is looked up automatically from the branch.

    Returns a dict suitable for passing to subprocess.run(env=...) or
    os.environ.update(...).
    """
    if w is None:
        w = get_workspace_client()

    # Resolve the endpoint once so we can reuse it for host + token
    ep = _get_primary_endpoint(project_id, branch_id, w=w)

    if host is None:
        host = ep.status.hosts.host
    token = get_lakebase_token(project_id, branch_id, w=w, _endpoint_name=ep.name)

    return {
        "PGPROJECTID": project_id,
        "PGBRANCHID": branch_id,
        "PGHOST": host,
        "PGPORT": str(port),
        "PGUSER": user or "",
        "PGPASSWORD": token,
        "PGDATABASE": database,
        "PGSSLMODE": sslmode,
    }


def get_psycopg_conn_params(pg_env: dict) -> dict:
    """Convert a PG env dict into psycopg2.connect() keyword arguments."""
    return {
        "host": pg_env["PGHOST"],
        "port": pg_env["PGPORT"],
        "user": pg_env["PGUSER"],
        "password": pg_env["PGPASSWORD"],
        "dbname": pg_env["PGDATABASE"],
        "sslmode": pg_env["PGSSLMODE"],
    }


def write_pgpass_file(pg_env: dict) -> str:
    """
    Write a temporary .pgpass file for CLI tools (psql, nominatim, osm2pgsql).

    Returns the path to the file.  Caller should set PGPASSFILE=<path> and
    unset PGPASSWORD so that child processes read from the file.
    """
    fd, path = tempfile.mkstemp(prefix="pgpass_")
    os.chmod(path, 0o600)
    entry = f"{pg_env['PGHOST']}:{pg_env['PGPORT']}:*:{pg_env['PGUSER']}:{pg_env['PGPASSWORD']}\n"
    with os.fdopen(fd, "w") as f:
        f.write(entry)
    return path


def refresh_pgpass(pgpass_path: str, pg_env: dict) -> str:
    """
    Refresh the token inside an existing pgpass file.

    Generates a new Lakebase token and atomically rewrites the pgpass file.
    Returns the new token.
    """
    new_token = get_lakebase_token(
        project_id=pg_env["PGPROJECTID"],
        branch_id=pg_env["PGBRANCHID"],
    )
    tmp_path = pgpass_path + ".tmp"
    entry = f"{pg_env['PGHOST']}:{pg_env['PGPORT']}:*:{pg_env['PGUSER']}:{new_token}\n"
    fd = os.open(tmp_path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    with os.fdopen(fd, "w") as f:
        f.write(entry)
    os.replace(tmp_path, pgpass_path)
    pg_env["PGPASSWORD"] = new_token
    return new_token


def run_cmd(cmd, env=None, **kwargs):
    """
    Run a shell command, inheriting the current environment plus optional overrides.

    Merges os.environ with *env* so that PATH and other system vars survive.
    """
    merged = {**os.environ, **(env or {})}
    if isinstance(cmd, str):
        kwargs.setdefault("shell", True)
    return subprocess.run(cmd, env=merged, check=True, **kwargs)